# Experimental NAFNet + full MambaIRv2 Base fusion

This notebook leaves `weights/best_model.pt` untouched. It trains an identity-safe spatial fusion gate between the accepted local NAFNet and the official full MambaIRv2 Base x2 model, then performs a D4/MSE PSNR-polish stage. The smaller Light variant remains available through `--mambair_variant light`.

In [ ]:
!nvidia-smi
!git clone --branch main https://github.com/kmbeddedd/semicon_2026.git
%cd /content/semicon_2026
!git lfs pull


## Install the project and official MambaIRv2
MambaIRv2 uses compiled selective-scan kernels. If Colab changes its CUDA/PyTorch image, install matching `causal-conv1d` and `mamba-ssm` wheels as described by the official repository.

In [ ]:
!pip install -q -r requirements.txt ninja einops timm basicsr
!git clone --depth 1 https://github.com/csguoh/MambaIR.git /content/MambaIR
!pip install -q causal-conv1d mamba-ssm --no-build-isolation
!wget -q -nc https://github.com/csguoh/MambaIR/releases/download/v1.0/mambairv2_classicSR_Base_x2.pth -O /content/mambairv2_classicSR_Base_x2.pth


## Mount Drive and extract the paired data
Place `train.zip` in the root of My Drive. Extraction is kept on `/content` so Drive I/O does not starve the GPU.

In [ ]:
import os, zipfile
from google.colab import drive
drive.mount('/content/drive')
archive = '/content/drive/MyDrive/train.zip' if os.path.exists('/content/drive/MyDrive/train.zip') else 'train.zip'
if not os.path.exists(archive):
    raise FileNotFoundError('Upload train.zip to My Drive or keep it in the repository')
os.makedirs('data', exist_ok=True)
with zipfile.ZipFile(archive) as source:
    source.extractall('data')


## Stage A: learn complementary fusion with ADD+
The 92.8 MB official MambaIRv2 Base branch is frozen initially. The NAFNet branch receives a very small learning rate, while the zero-initialized fusion gate learns ten times faster. Auto-batching measures the complete dual-branch path and conservatively targets 82% of available VRAM on a 15 GB T4.

In [ ]:
import os, subprocess, torch
EXPERIMENT_DIR = '/content/drive/MyDrive/semicon_mambair_base_fusion'
LATEST = os.path.join(EXPERIMENT_DIR, 'latest_model.pt')
os.makedirs(EXPERIMENT_DIR, exist_ok=True)
stage_a = [
    'python', 'train_fusion.py',
    '--mambair_repo', '/content/MambaIR', '--mambair_variant', 'base',
    '--save_dir', EXPERIMENT_DIR, '--epochs', '12', '--warmup_epochs', '1',
    '--batch_size', '1', '--auto_batch_size', '--target_vram_fraction', '0.82', '--max_batch_size', '8',
    '--num_workers', '2', '--no_cache', '--augmentation', 'add_plus', '--add_probability', '0.5',
    '--local_lr', '2e-6', '--fusion_lr', '2e-5', '--global_lr', '1e-6',
    '--freeze_global', '--no-freeze_local', '--psnr_polish_epochs', '0', '--seed', '42'
]
last_epoch = int(torch.load(LATEST, map_location='cpu').get('epoch', 0)) if os.path.exists(LATEST) else 0
if last_epoch < 12:
    if os.path.exists(LATEST):
        stage_a += ['--resume', LATEST]
    else:
        stage_a += ['--local_weights', 'weights/best_model.pt', '--global_weights', '/content/mambairv2_classicSR_Base_x2.pth']
    subprocess.run(stage_a, check=True)
else:
    print(f'Stage A already complete at epoch {last_epoch}.')


## Stage B: in-distribution D4 and pure-MSE polish
This extends the same run to epoch 20, removes synthetic mixing, and linearly anneals the objective to exact MSE. Validation still selects every epoch by full-image PSNR.

In [ ]:
stage_b = [
    'python', 'train_fusion.py', '--resume', LATEST,
    '--mambair_repo', '/content/MambaIR', '--mambair_variant', 'base', '--save_dir', EXPERIMENT_DIR,
    '--epochs', '20', '--warmup_epochs', '1', '--batch_size', '1',
    '--auto_batch_size', '--target_vram_fraction', '0.82', '--max_batch_size', '8',
    '--num_workers', '2', '--no_cache', '--augmentation', 'd4',
    '--local_lr', '2e-6', '--fusion_lr', '2e-5', '--global_lr', '1e-6',
    '--freeze_global', '--no-freeze_local', '--psnr_polish_epochs', '8', '--seed', '42'
]
last_epoch = int(torch.load(LATEST, map_location='cpu').get('epoch', 0))
if last_epoch < 20:
    subprocess.run(stage_b, check=True)
else:
    print(f'Stage B already complete at epoch {last_epoch}.')


## Evaluate only if fusion beat the accepted checkpoint

In [ ]:
BEST = os.path.join(EXPERIMENT_DIR, 'best_model.pt')
if os.path.exists(BEST):
    subprocess.run([
        'python', 'eval.py', '--input_dir', 'data/val/NoisyLR', '--target_dir', 'data/val/GT',
        '--output_dir', '/content/fusion_val', '--weights', BEST, '--mambair_repo', '/content/MambaIR',
        '--scale', '2', '--batch_size', '1'
    ], check=True)
else:
    print('No best_model.pt was written: fusion did not beat the accepted local checkpoint.')
